# TRACT: From Zero-Shot to Expert-Reviewed Security Crosswalk

**A practitioner's guide to mapping AI security frameworks using machine learning**

This notebook tells the story of how we built a system that reads any security control and tells you which part of a universal security taxonomy it belongs to. We'll walk through every experiment, every failure, and every hard-won insight — in plain language.

---

In [ ]:
"""Setup: imports, seeds, prerequisite checks."""
import sys
import json
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.manifold import TSNE

# Ensure nb_helpers is importable from notebooks/
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))

from nb_helpers import (
    PROJECT_ROOT, RESULTS_DIR, DATA_DIR,
    PHASE0_DIR, PHASE1B_DIR, PHASE1C_DIR,
    REVIEW_DIR, BRIDGE_DIR, DATASET_DIR,
    OKABE_ITO, SEQUENTIAL_BLUE, SEQUENTIAL_ORANGE, DIVERGING,
    FigureCounter, style_axes, plotly_with_fallback,
    load_phase0_experiment, load_firewalled_baseline,
    load_fold_metrics, load_training_logs,
    load_calibration_data, load_deployment_embeddings,
    load_review_metrics, load_review_export,
    load_cre_hierarchy, load_crosswalk, load_framework_metadata,
    check_prerequisites,
)

# Reproducibility
random.seed(42)
np.random.seed(42)

# Matplotlib defaults
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "savefig.bbox": "tight",
})
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# Figure counter — instantiate once
fig_counter = FigureCounter()

# Prerequisite check
missing = check_prerequisites()
if missing:
    print("⚠️  Missing prerequisite files:")
    for m in missing:
        print(f"   - {m}")
    print("\nSee the notebook README for setup instructions.")
else:
    print("✓ All prerequisite files found")

# Verify CWD
cwd = Path.cwd()
if cwd.name != "notebooks":
    print(f"⚠️  Expected CWD = notebooks/, got {cwd.name}/. Shell cells may not work correctly.")
else:
    print(f"✓ CWD: {cwd}")

print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✓ NumPy {np.__version__}, Matplotlib {matplotlib.__version__}")

## 1. Why This Exists: The Security Framework Translation Problem

You're a security architect. Your organization runs workloads across three cloud providers, 
deploys AI models in production, and needs to demonstrate compliance with half a dozen 
frameworks — NIST 800-53, ISO 27001, OWASP, MITRE ATLAS, the EU AI Act, and more.

Each framework has its own taxonomy, its own control numbering, its own way of saying 
"encrypt data at rest." Comparing them manually means reading thousands of controls 
and building mental bridges between different vocabularies for the same concepts.

This is the **N² problem**: with *k* frameworks, you need *k(k−1)/2* pairwise mappings.

### The Common Requirements Enumeration (CRE)

The [CRE project](https://opencre.org) solves this by creating a **shared coordinate system** 
for security requirements. Think of it as GPS coordinates for security concepts — instead of 
describing locations relative to each other ("the bakery is two blocks from the bank"), 
you assign each location a universal coordinate.

CRE organizes 522 security concept **hubs** into a hierarchy. Every security control from 
every framework can be mapped to one or more hubs. Once mapped, cross-framework comparison 
becomes a simple lookup: "which controls share the same hub?"

Let's look at this hierarchy:

In [ ]:
hierarchy = load_cre_hierarchy()
hubs = hierarchy["hubs"]
print(f"CRE hierarchy: {len(hubs)} hubs")

In [ ]:
ids, names, parents = [], [], []
for hub_id, hub in sorted(hubs.items()):
    ids.append(hub_id)
    names.append(hub["name"])
    parents.append(hub.get("parent_id") or "")

fig = go.Figure(go.Sunburst(
    ids=ids,
    labels=names,
    parents=parents,
    maxdepth=3,
))
fig_num = fig_counter.next(1)
plotly_with_fallback(fig, fig_num, "CRE Hub Taxonomy — 522 Hubs")

The sunburst above shows the full CRE taxonomy. Click to drill down into any branch. 
The outer rings show increasingly specific security concepts — from broad areas like 
"Authentication" and "Cryptography" down to specific requirements like 
"Password complexity" or "Key rotation."

This is our target space: every security control we encounter will be mapped to one of 
these 522 hubs.

### The Assignment Paradigm

TRACT uses an **assignment paradigm**: instead of comparing controls to each other, 
we train a model to map each control independently to the CRE hierarchy:

$$g(\text{control\_text}) \rightarrow \text{CRE\_hub}$$

This is fundamentally different from pairwise comparison. Here's why it matters:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: N² pairwise
n_fw = 6
for i in range(n_fw):
    for j in range(i+1, n_fw):
        ax1.plot([i, j], [0, 0], 'o-', color=OKABE_ITO[4], alpha=0.3, markersize=8)
ax1.set_xlim(-0.5, n_fw-0.5)
ax1.set_title(f"Pairwise: {n_fw*(n_fw-1)//2} comparisons", fontsize=12)
ax1.set_yticks([])

# Right: Assignment paradigm
for i in range(n_fw):
    ax2.annotate("", xy=(3, 2-i*0.7), xytext=(i, -2),
                 arrowprops=dict(arrowstyle="->", color=OKABE_ITO[i % len(OKABE_ITO)]))
ax2.set_title(f"Assignment: {n_fw} mappings", fontsize=12)
ax2.set_yticks([])

fig_num = fig_counter.next(1)
fig.suptitle(f"{fig_num}: Why Assignment Scales", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
plt.close(fig)

With 6 frameworks, pairwise comparison requires 15 separate mappings. Add a 7th framework 
and you need 21 — every new framework means comparing against *all* existing ones.

The assignment paradigm? Just 6 mappings (now 7). Each framework maps to the shared 
coordinate system independently. Cross-framework comparison is then a free side effect 
of the mapping — frameworks that map to the same hub are, by definition, addressing the 
same security concept.

### What This Notebook Covers

1. **Data Landscape** — What we're working with: 31 frameworks, 5,238 assignments, 522 hubs
2. **Phase 0: Zero-Shot Baselines** — How well do off-the-shelf models do?
3. **Model Selection** — Choosing the right base embedding model
4. **Contrastive Fine-Tuning** — Teaching the model what "same hub" means
5. **Ablation Analysis** — What actually helps (and what doesn't)
6. **Hub Firewall** — How we keep evaluation honest
7. **Final Results** — Aggregate performance with bootstrap confidence intervals
8. **Error Analysis** — Where the model struggles and why
9. **Calibration** — Can we trust the confidence scores?
10. **Human Review** — Expert validation of 5,238 assignments
11. **CLI Tutorial** — Using TRACT in practice
12. **Retrospective** — What we built, what we learned, what's next

> **Plain English:** We built a system that reads any security control and tells you 
> which part of a universal security taxonomy it belongs to. This lets you instantly 
> compare any two frameworks — without reading thousands of controls manually.